# Notebook 01 · Q-Learning: el agente aprende el camino solo

**Introducción al Deep Learning — Módulo 4, Unidad 2: Aprendizaje por refuerzo (Parte II)**

En la Parte I calculamos los valores V(s) **conociendo** el entorno (teníamos la función `paso`).
Eso es hacer trampa: en un problema real el agente **no sabe** cómo funciona el mundo.

El **Q-Learning** resuelve esto. El agente:

1. Mantiene una **tabla Q** con una casilla por cada par (estado, acción).
2. **Actúa** en el entorno y observa la recompensa y el nuevo estado.
3. **Ajusta** la casilla usada acercándola a lo observado (regla de Bellman).

Repitiendo esto miles de veces, la tabla converge y aparece la política óptima. Todo con **NumPy**,
en segundos.

> 💡 Es *aprendizaje sin modelo* (model-free): el agente solo ve estados y recompensas, nunca las reglas.

## 1. El entorno (el agente NO lo conoce)

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
np.random.seed(0)

FILAS, COLS = 4, 4
META = (3, 3)
TRAMPAS = [(1, 1), (2, 3)]
ACCIONES = [0, 1, 2, 3]               # arriba, abajo, izquierda, derecha
MOV = {0: (-1, 0), 1: (1, 0), 2: (0, -1), 3: (0, 1)}
FLECHAS = {0: "^", 1: "v", 2: "<", 3: ">"}

def es_terminal(s):
    return s == META or s in TRAMPAS

def paso(s, a):
    df, dc = MOV[a]
    f = min(max(s[0] + df, 0), FILAS - 1)
    c = min(max(s[1] + dc, 0), COLS - 1)
    ns = (f, c)
    if ns == META:    return ns, 10.0, True
    if ns in TRAMPAS: return ns, -10.0, True
    return ns, -1.0, False

# El agente solo puede LLAMAR a paso(); no ve su código.
print("Entorno preparado (rejilla 4x4).")

## 2. La tabla Q

Un array de forma `(filas, columnas, acciones)`. Empieza **a cero**: el agente no sabe nada.

In [ ]:
Q = np.zeros((FILAS, COLS, len(ACCIONES)))
print("Forma de la tabla Q:", Q.shape, "->", Q.size, "valores, todos a 0")

## 3. Exploración vs. explotación (ε-greedy)

En cada paso el agente decide:

- con probabilidad **ε** → **explora**: acción al azar (descubre cosas nuevas),
- con probabilidad **1−ε** → **explota**: la mejor acción según Q (aprovecha lo que sabe).

ε empieza **alto** (todo por explorar) y **baja** con el tiempo (ya sabe lo que hace).

In [ ]:
def elegir_accion(s, epsilon):
    if np.random.rand() < epsilon:
        return np.random.choice(ACCIONES)     # explorar
    return int(np.argmax(Q[s[0], s[1]]))      # explotar

## 4. El bucle de aprendizaje

Esta es la regla del Q-Learning, aplicada tras cada acción:

$$Q(s,a) \leftarrow Q(s,a) + \alpha\,[\,r + \gamma\max_{a'}Q(s',a') - Q(s,a)\,]$$

- **α** (tasa de aprendizaje): cuánto caso hacemos a la nueva experiencia.
- **γ** (descuento): cuánto pesa el futuro.
- El corchete es el **error**: diferencia entre lo observado y lo que teníamos estimado.

In [ ]:
ALPHA, GAMMA = 0.1, 0.9
EPISODIOS = 2000
eps_ini, eps_fin = 1.0, 0.05

recompensas_por_ep = []
for ep in range(EPISODIOS):
    epsilon = max(eps_fin, eps_ini * (1 - ep / EPISODIOS))   # decae linealmente
    s = (0, 0)
    total = 0.0
    for _ in range(100):
        a = elegir_accion(s, epsilon)
        ns, r, fin = paso(s, a)
        # --- regla de Q-Learning ---
        mejor_futuro = 0.0 if fin else np.max(Q[ns[0], ns[1]])
        objetivo = r + GAMMA * mejor_futuro
        Q[s[0], s[1], a] += ALPHA * (objetivo - Q[s[0], s[1], a])
        # ---------------------------
        total += r
        s = ns
        if fin:
            break
    recompensas_por_ep.append(total)

print("Entrenamiento terminado tras", EPISODIOS, "episodios.")
print(f"Recompensa media primeros 100 ep: {np.mean(recompensas_por_ep[:100]):+.2f}")
print(f"Recompensa media últimos  100 ep: {np.mean(recompensas_por_ep[-100:]):+.2f}")

### La curva de aprendizaje

In [ ]:
def media_movil(x, n=50):
    return np.convolve(x, np.ones(n) / n, mode="valid")

plt.figure(figsize=(7, 3.2))
plt.plot(media_movil(recompensas_por_ep), color="#000F9F")
plt.axhline(0, color="grey", linewidth=0.8)
plt.title("Recompensa por episodio (media móvil): el agente mejora")
plt.xlabel("episodio"); plt.ylabel("recompensa"); plt.show()

La curva sube y se estabiliza: el agente pasa de perder puntos (cae en trampas, da vueltas)
a llegar a la meta de forma eficiente. **Y nadie le dijo el camino.**

## 5. ¿Qué ha aprendido? La política

En cada casilla, la mejor acción es `argmax` de su fila en Q. Vamos a dibujarlo.

In [ ]:
V = Q.max(axis=2)                       # valor de cada estado = mejor Q
politica = Q.argmax(axis=2)

fig, ax = plt.subplots(figsize=(4.4, 4.4))
ax.imshow(V, cmap="RdYlGn")
for f in range(FILAS):
    for c in range(COLS):
        if (f, c) == META:      ax.text(c, f, "META", ha="center", va="center", fontsize=8)
        elif (f, c) in TRAMPAS: ax.text(c, f, "X", ha="center", va="center", fontsize=14)
        else:                   ax.text(c, f, FLECHAS[politica[f, c]], ha="center", va="center", fontsize=20)
ax.set_xticks(range(COLS)); ax.set_yticks(range(FILAS))
ax.set_title("Política aprendida (color = valor del estado)")
plt.show()

### Probamos la política aprendida

In [ ]:
def jugar(mostrar=False):
    s, total, camino = (0, 0), 0.0, [(0, 0)]
    for _ in range(50):
        a = int(np.argmax(Q[s[0], s[1]]))
        s, r, fin = paso(s, a)
        total += r; camino.append(s)
        if fin: break
    if mostrar:
        print("Camino:", camino)
    return total

print("Recompensa siguiendo la política aprendida:", jugar(mostrar=True))
print("Media en 200 partidas:", np.mean([jugar() for _ in range(200)]))

## Experimenta tú

1. Pon `EPISODIOS = 200`. ¿Le da tiempo a aprender? ¿Y con `50`?
2. Fija `epsilon = 0` desde el principio (nunca explora). ¿Aprende el camino? ¿Por qué no?
3. Cambia `ALPHA` a `0.9`. ¿La curva es más rápida pero más ruidosa?
4. Mueve la meta o añade otra trampa y reentrena.

## Resumen

- **Q-Learning** aprende la tabla Q **actuando**, sin conocer el entorno (model-free).
- **ε-greedy** equilibra explorar (probar) y explotar (aprovechar); ε baja con el tiempo.
- La **regla de actualización** acerca Q(s,a) al objetivo `r + γ·max Q(s')`.
- Al final, la política óptima es simplemente `argmax` de la tabla en cada estado.

⚠️ Esto funciona porque hay **16 estados**. Con millones de estados (p. ej. los píxeles de un videojuego)
la tabla no cabe. La solución es sustituirla por una **red neuronal**: el **DQN** del Notebook 02.